# Bonus — When `dynamo=True` actually pays off

In `Mini_ONNX`, we used `dynamo=False` because SceneSeg is a plain CNN — both exporters produce essentially identical ONNX. That's the boring case. Most people read about `dynamo=True` and wonder why it exists at all.

This notebook answers the question on a model where the difference shows up clearly: **SegFormer-B0**, an attention-based semantic segmentation model from NVIDIA, pretrained on Cityscapes.

We'll export the same model two ways, count nodes, compare ops, and benchmark inference. The numbers do the talking.


## Setup


In [ ]:
!pip install transformers onnxruntime onnx onnxscript -q


In [ ]:
import torch
import torch.nn as nn
import numpy as np
import time, os, urllib.request
from PIL import Image
import matplotlib.pyplot as plt
import onnx
import onnxruntime as ort
from transformers import SegformerForSemanticSegmentation, SegformerImageProcessor

print(f"PyTorch {torch.__version__} | ORT {ort.__version__} | ONNX {onnx.__version__}")


In [ ]:
# Cityscapes-style 19-class color palette + helper for segmentation visualization
CITYSCAPES_PALETTE = np.array([
    [128,  64, 128],   # 0  road
    [244,  35, 232],   # 1  sidewalk
    [ 70,  70,  70],   # 2  building
    [102, 102, 156],   # 3  wall
    [190, 153, 153],   # 4  fence
    [153, 153, 153],   # 5  pole
    [250, 170,  30],   # 6  traffic light
    [220, 220,   0],   # 7  traffic sign
    [107, 142,  35],   # 8  vegetation
    [152, 251, 152],   # 9  terrain
    [ 70, 130, 180],   # 10 sky
    [220,  20,  60],   # 11 person
    [255,   0,   0],   # 12 rider
    [  0,   0, 142],   # 13 car
    [  0,   0,  70],   # 14 truck
    [  0,  60, 100],   # 15 bus
    [  0,  80, 100],   # 16 train
    [  0,   0, 230],   # 17 motorcycle
    [119,  11,  32],   # 18 bicycle
], dtype=np.uint8)


def logits_to_color(logits, target_wh):
    """Convert (1, C, H, W) logits to an (H_t, W_t, 3) RGB segmentation image."""
    import cv2
    seg = np.argmax(logits[0], axis=0).astype(np.uint8)
    rgb = CITYSCAPES_PALETTE[np.clip(seg, 0, 18)]
    return cv2.resize(rgb, target_wh, interpolation=cv2.INTER_NEAREST)


## Load SegFormer-B0 (Cityscapes)

Pretrained on Cityscapes — same urban driving domain as your Static_Quantization workshop. ~14M params, attention-everywhere architecture (hierarchical Transformer encoder + lightweight MLP decoder). One line from HuggingFace.


In [ ]:
CKPT = "nvidia/segformer-b0-finetuned-cityscapes-512-1024"

model     = SegformerForSemanticSegmentation.from_pretrained(CKPT)
processor = SegformerImageProcessor.from_pretrained(CKPT)
model.eval()

n_params = sum(p.numel() for p in model.parameters()) / 1e6
print(f"SegFormer-B0:  {n_params:.1f}M params")
print(f"Input size:    {processor.size}")
print(f"Num classes:   {model.config.num_labels}")


## Get a Cityscapes-style test image

We'll use the same Waymo frames as the rest of the course (driving scenes match Cityscapes domain well enough for this demo).


In [ ]:
!mkdir -p /content/data
!wget -qq https://optical-flow-data.s3.eu-west-3.amazonaws.com/waymo_images.zip -O /content/data/waymo.zip
!unzip -qq -o /content/data/waymo.zip -d /content/data/

import glob
frame_path = sorted(glob.glob('/content/data/night/front_images_night/*.jpg'))[100]
img = Image.open(frame_path).convert("RGB")

inputs = processor(images=img, return_tensors="pt")
x      = inputs["pixel_values"]              # (1, 3, 512, 1024)
x_np   = x.numpy()
print(f"Input tensor: {tuple(x.shape)}")

plt.figure(figsize=(10, 4))
plt.imshow(img.resize((1024, 512))); plt.axis('off'); plt.title('Input frame'); plt.show()


## Wrap & run as PyTorch

HuggingFace models return a dataclass — for ONNX export we want a plain tensor output. Tiny wrapper:


In [ ]:
class ExportableSegFormer(nn.Module):
    def __init__(self, base):
        super().__init__()
        self.base = base
    def forward(self, pixel_values: torch.Tensor) -> torch.Tensor:
        return self.base(pixel_values=pixel_values).logits   # (1, C, H/4, W/4)

wrapped = ExportableSegFormer(model).eval()

with torch.no_grad():
    pt_out = wrapped(x).numpy()
print(f"PyTorch output: {pt_out.shape}")

# Visualize: input | PyTorch segmentation overlay
import numpy as np
input_img  = np.array(img.resize((1024, 512)))
seg_color  = logits_to_color(pt_out, (1024, 512))
overlay    = (input_img * 0.5 + seg_color * 0.5).astype(np.uint8)

fig, axes = plt.subplots(1, 2, figsize=(16, 4))
axes[0].imshow(input_img);  axes[0].set_title('Input frame');                    axes[0].axis('off')
axes[1].imshow(overlay);    axes[1].set_title('PyTorch — Cityscapes segmentation'); axes[1].axis('off')
plt.tight_layout(); plt.show()


## Export with `dynamo=False` (legacy)

The classic path. Tracing-based — runs the model once and records every tensor op.


In [ ]:
import warnings
warnings.filterwarnings("ignore")

torch.onnx.export(
    wrapped, x,
    '/content/segformer_legacy.onnx',
    opset_version=17,
    input_names=['pixel_values'],
    output_names=['logits'],
    dynamo=False,
)
sz = os.path.getsize('/content/segformer_legacy.onnx') / 1e6
print(f"Legacy:  {sz:.1f} MB")


## Export with `dynamo=True` (modern)

Uses `torch.export` + Torch FX. Captures the model at the bytecode level — fused attention ops stay fused.


In [ ]:
torch.onnx.export(
    wrapped, (x,),
    '/content/segformer_dynamo.onnx',
    opset_version=18,
    input_names=['pixel_values'],
    output_names=['logits'],
    dynamo=True,
)
sz = os.path.getsize('/content/segformer_dynamo.onnx') / 1e6
print(f"Dynamo:  {sz:.1f} MB")


In [ ]:
!ls -lh /content/segformer_*.onnx /content/segformer_*.data 2>/dev/null || echo "(no .data sidecar files)"

print()
print("If you see .data files alongside .onnx, those hold the weights.")
print("Dynamo often externalizes them — that's why segformer_dynamo.onnx is small.")
print("ORT loads them automatically as long as both files stay together.")


## Side-by-side: what's inside each file?

ONNX is just a graph of nodes (operators). Let's count them.


In [ ]:
import os, glob

def graph_stats(path):
    m = onnx.load(path)
    counts = {}
    for n in m.graph.node:
        counts[n.op_type] = counts.get(n.op_type, 0) + 1

    onnx_mb = os.path.getsize(path) / 1e6
    # Sidecar .data files (where dynamo often stores weights)
    base = os.path.basename(path)
    sidecars = [f for f in os.listdir(os.path.dirname(path) or '.')
                if f.startswith(base) and f != base]
    data_mb = sum(os.path.getsize(os.path.join(os.path.dirname(path) or '.', f)) / 1e6
                  for f in sidecars)

    return {
        'onnx_mb':  onnx_mb,
        'data_mb':  data_mb,
        'total_mb': onnx_mb + data_mb,
        'total':    sum(counts.values()),
        'unique':   len(counts),
        'op_counts': counts,
    }

legacy = graph_stats('/content/segformer_legacy.onnx')
dynamo = graph_stats('/content/segformer_dynamo.onnx')

print(f"{'':16} {'Legacy':>10} {'Dynamo':>10} {'Δ':>10}")
print('-' * 50)
print(f"{'.onnx file':16} {legacy['onnx_mb']:>9.1f} MB {dynamo['onnx_mb']:>9.1f} MB "
      f"{(dynamo['onnx_mb']/legacy['onnx_mb']-1)*100:>+9.1f}%")
print(f"{'.data sidecar':16} {legacy['data_mb']:>9.1f} MB {dynamo['data_mb']:>9.1f} MB")
print(f"{'TOTAL deploy':16} {legacy['total_mb']:>9.1f} MB {dynamo['total_mb']:>9.1f} MB "
      f"{(dynamo['total_mb']/legacy['total_mb']-1)*100:>+9.1f}%")
print('-' * 50)
print(f"{'Total nodes':16} {legacy['total']:>10} {dynamo['total']:>10} "
      f"{(dynamo['total']/legacy['total']-1)*100:>+9.1f}%")
print(f"{'Unique op types':16} {legacy['unique']:>10} {dynamo['unique']:>10}")


### The attention fusion story

The big difference is in the attention blocks. Legacy explodes each attention layer into the underlying primitives (`MatMul`, `Softmax`, `Reshape`, `Transpose`, …). Dynamo keeps them as fused ops where possible.


In [ ]:
attention_primitives = {'MatMul', 'Softmax', 'Reshape', 'Transpose', 'Cast'}

def primitives(stats):
    return sum(c for op, c in stats['op_counts'].items() if op in attention_primitives)

print(f"Attention primitives in legacy: {primitives(legacy):>4}")
print(f"Attention primitives in dynamo: {primitives(dynamo):>4}")
print()

# Top 8 ops in each
for label, stats in [('LEGACY', legacy), ('DYNAMO', dynamo)]:
    top = sorted(stats['op_counts'].items(), key=lambda kv: -kv[1])[:8]
    print(f"\n--- Top 8 ops in {label} ---")
    for op, n in top:
        print(f"  {op:<25} {n}")


## Numerical equivalence — same math, different graph

If both exporters did their job, both ONNX files should produce the same output (down to floating-point reordering).


In [ ]:
sess_legacy = ort.InferenceSession('/content/segformer_legacy.onnx', providers=['CPUExecutionProvider'])
sess_dynamo = ort.InferenceSession('/content/segformer_dynamo.onnx', providers=['CPUExecutionProvider'])

out_legacy = sess_legacy.run(None, {'pixel_values': x_np})[0]
out_dynamo = sess_dynamo.run(None, {'pixel_values': x_np})[0]

print(f"PyTorch vs Legacy ONNX:  max diff {np.abs(pt_out - out_legacy).max():.2e}")
print(f"PyTorch vs Dynamo ONNX:  max diff {np.abs(pt_out - out_dynamo).max():.2e}")
print(f"Legacy  vs Dynamo ONNX:  max diff {np.abs(out_legacy - out_dynamo).max():.2e}")
print(f"\nAll three within 1e-3? {all(np.allclose(a, b, atol=1e-3) for a, b in [(pt_out, out_legacy), (pt_out, out_dynamo), (out_legacy, out_dynamo)])}")


### Visual check — do all three produce the same segmentation?

Numbers said yes (max diff < 1e-3). Eyes should agree.


In [ ]:
# Visual sanity: do all three produce the same segmentation map?
fig, axes = plt.subplots(1, 4, figsize=(22, 4))
axes[0].imshow(input_img);                                axes[0].set_title('Input');                     axes[0].axis('off')
axes[1].imshow(logits_to_color(pt_out,    (1024, 512))); axes[1].set_title('PyTorch');                   axes[1].axis('off')
axes[2].imshow(logits_to_color(out_legacy,(1024, 512))); axes[2].set_title('Legacy ONNX');               axes[2].axis('off')
axes[3].imshow(logits_to_color(out_dynamo,(1024, 512))); axes[3].set_title('Dynamo ONNX');               axes[3].axis('off')
plt.tight_layout(); plt.show()


## Inference speed — does the cleaner graph actually run faster?

Both ONNX files are loaded by the same ONNX Runtime with the same providers. The only variable is the graph. Let's time them.


In [ ]:
from tqdm.auto import tqdm

def bench(sess, label, n=10):
    """Warmup + n timed runs with a visible progress bar."""
    sess.run(None, {'pixel_values': x_np})
    ts = []
    for _ in tqdm(range(n), desc=label, leave=False):
        t0 = time.perf_counter()
        sess.run(None, {'pixel_values': x_np})
        ts.append((time.perf_counter() - t0) * 1000)
    return np.array(ts)

t_legacy = bench(sess_legacy, 'Legacy ONNX')
t_dynamo = bench(sess_dynamo, 'Dynamo ONNX')

print(f"\nLegacy ONNX:  {t_legacy.mean():>6.1f} ms  ±{t_legacy.std():.1f}")
print(f"Dynamo ONNX:  {t_dynamo.mean():>6.1f} ms  ±{t_dynamo.std():.1f}")
print(f"\nDynamo is {t_legacy.mean()/t_dynamo.mean():.2f}x the speed of legacy")
print("(>1 = dynamo wins, <1 = legacy wins, ≈1 = no real difference)")


---
## What you just saw

On a transformer model, exported through Colab CPU + ONNX Runtime, the result was:
1. **Dynamo's `.onnx` is much smaller** — but most of that win is because dynamo externalizes weights into a sidecar `.data` file, not because the graph itself is dramatically lighter.
2. **Dynamo's graph has fewer nodes** — the 172 `Constant` nodes that legacy inlines become proper initializers in dynamo. Cleaner representation, real structural win.
3. **Numerical output is identical** — both exporters preserve the math (~1e-5 noise).
4. **Speed didn't necessarily win.** On ORT-CPU you may see dynamo equal or slower than legacy. ORT's CPU fuser was tuned against legacy graphs for years; dynamo's cleaner output is *structurally* better but the runtime hasn't caught up. **On ORT-CUDA or TensorRT the picture often flips** — those runtimes do more aggressive fusion of dynamo's graph.

The real lesson is honest: **dynamo's wins are real but not universal**. Always benchmark on your actual deployment runtime.

### Practical guidance

- **Plain CNN, fixed input** → either works, legacy is simpler.
- **Transformer / attention-heavy model** → try `dynamo=True` first. Smaller file, cleaner graph, **but benchmark before assuming it's faster**.
- **Variable input shapes** → use `dynamo=True` with `Dim` objects.
- **Custom CUDA ops or research-grade code (BEVFormer, etc.)** → expect pain regardless; the export itself becomes a project.
- **The deploy target matters more than the export flag** — TensorRT will fuse most of what either exporter produces. ORT-CPU will not.

Pick the exporter that fits your model, then **measure on the runtime you'll actually deploy on**. Not on the runtime that's convenient for your tutorial.
